# （參考解答）03_probability_simulation_lln_clt

> 這是對應主 notebook 的**完整參考解答版**。建議先自己完成主notebook 的練習，再對照本檔。所有解說與解答皆為本專案原創。

# Week 3 — 機率複習：模擬、LLN 與 CLT

> 本 notebook 屬於「量化數學路線圖」（quant-math-roadmap）開源教學專案。
> 僅供**教育與研究方法論**用途，**不構成投資建議**，任何結果都不代表實際可獲利或可投資的策略。

## 學習目標

- 模擬常見分布並由樣本估計動差。
- 用模擬視覺化大數法則 (LLN)。
- 用模擬視覺化中央極限定理 (CLT)。
- 把抽樣不確定性連結到「估計平均策略報酬」。

## 預估學習時間

約 7–9 小時。

## 先備概念

- 隨機變數、期望值與變異數
- 基本 numpy

## 外部學習資源

- [MIT OpenCourseWare 18.05 Introduction to Probability and Statistics](https://ocw.mit.edu/courses/18-05-introduction-to-probability-and-statistics-spring-2022/)

> 外部資源僅供參考連結；本專案不重製任何受版權保護的課程材料。

In [ ]:
# 教學樣式設定（CJK 字型、負號正常顯示、固定隨機種子）
import matplotlib as _mpl
_mpl.rcParams['font.sans-serif'] = [
    'PingFang TC', 'Heiti TC', 'Microsoft JhengHei',
    'Noto Sans CJK TC', 'Noto Sans TC',
    'WenQuanYi Zen Hei', 'Source Han Sans TC',
    'Arial Unicode MS', 'DejaVu Sans',
]
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## 概念說明

### 大數法則 (Law of Large Numbers, LLN)

當樣本數 $n$ 增加，樣本平均 $\bar X_n$ 會收斂到真實期望值 $\mu$：

$$ \bar X_n = \frac1n\sum_{i=1}^n X_i \xrightarrow[n\to\infty]{} \mu. $$

### 中央極限定理 (Central Limit Theorem, CLT)

不論母體分布形狀如何，樣本平均的**抽樣分布**會趨近常態：

$$ \frac{\bar X_n - \mu}{\sigma/\sqrt n} \xrightarrow{d} N(0, 1). $$

LLN 告訴我們平均**會收斂到哪**；CLT 告訴我們在有限 $n$ 下平均的**不確定性有多大**（標準誤 $\sigma/\sqrt n$）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.math.probability import (
    empirical_moments, running_mean,
    sampling_distribution_of_mean, simulate_normal,
)

rng = np.random.default_rng(2024)

### 模擬分布並估計動差

In [ ]:
normal_draws = simulate_normal(mean=0.001, std=0.02, size=10_000, seed=1)
moments = empirical_moments(normal_draws)
print('估計動差:', {k: round(v, 6) for k, v in moments.items()})
print('真實 mean = 0.001, 真實 std = 0.02')

### 視覺化 LLN：樣本平均的收斂

In [ ]:
samples = simulate_normal(mean=0.05, std=1.0, size=20_000, seed=3)
path = running_mean(samples)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(1, len(path) + 1), path, label='累積樣本平均')
ax.axhline(0.05, linestyle='--', label='真實期望值 = 0.05')
ax.set_title('大數法則：樣本平均隨樣本數收斂')
ax.set_xlabel('樣本數 n')
ax.set_ylabel('累積平均')
ax.legend()
plt.show()

曲線一開始劇烈震盪，隨 $n$ 增加逐漸穩定到真實期望值。**前段的震盪正是抽樣不確定性**——這也是為什麼短期回測的平均報酬不可盡信。

### 視覺化 CLT：樣本平均的抽樣分布

In [ ]:
small = sampling_distribution_of_mean(
    population_sampler='exponential', sample_size=5,
    n_experiments=5000, seed=4)
large = sampling_distribution_of_mean(
    population_sampler='exponential', sample_size=200,
    n_experiments=5000, seed=4)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].hist(small, bins=40)
axes[0].set_title('樣本數 n=5 的樣本平均分布')
axes[0].set_xlabel('樣本平均')
axes[0].set_ylabel('次數')
axes[1].hist(large, bins=40)
axes[1].set_title('樣本數 n=200 的樣本平均分布')
axes[1].set_xlabel('樣本平均')
axes[1].set_ylabel('次數')
plt.tight_layout()
plt.show()
print('n=5 標準差:', round(float(np.std(small)), 4))
print('n=200 標準差:', round(float(np.std(large)), 4))

母體是**指數分布**（高度右偏），但樣本平均的分布隨 $n$ 增加越來越像常態，且越來越集中（標準誤縮小）。這就是 CLT。

### 連結到策略報酬

把「一檔策略每天的報酬」想成隨機變數。我們真正想知道的是它的**真實期望報酬 $\mu$**，但只能用有限樣本的樣本平均去估計。CLT 告訴我們：樣本平均的不確定性是 $\sigma/\sqrt n$——報酬波動越大、資料越少，這個估計就越不可靠。

In [ ]:
# 一檔「真實期望報酬為 0」的策略，只是運氣好
strategy = simulate_normal(mean=0.0, std=0.01, size=252, seed=99)
mean_est = strategy.mean()
se = strategy.std(ddof=1) / np.sqrt(len(strategy))
print(f'一年資料的樣本平均日報酬 = {mean_est:.6f}')
print(f'標準誤 = {se:.6f}')
print('樣本平均看起來不為 0，但這完全可能只是抽樣雜訊。')

## 練習

請依序完成以下練習。**基礎練習**鞏固定義，**應用練習**動手寫程式，**反思問題**把數學連結到回測與研究方法論。

> 主 notebook 的程式練習提供可執行的起始碼（starter）。完整參考解答請見 `notebooks/solutions/` 對應的 `_solution` notebook。

### 基礎練習

1. 用自己的話說明 LLN 與 CLT 各自回答了什麼問題。
2. 標準誤 $\sigma/\sqrt n$ 中，要讓標準誤減半需要多少倍的樣本？
3. 為什麼母體不是常態，樣本平均仍可能接近常態？

### 應用練習

In [ ]:
# 應用練習 1：模擬 50000 次擲一枚公正硬幣，計算正面比例的累積平均，
# 並確認它收斂到 0.5。
flips = rng.integers(0, 2, size=50_000).astype(float)
coin_path = running_mean(flips)
print('最終累積比例:', round(float(coin_path[-1]), 4))
assert abs(coin_path[-1] - 0.5) < 0.02

In [ ]:
# 應用練習 2：對 sample_size = 2, 10, 50, 250 各做 4000 次實驗，
# 印出樣本平均的標準差，觀察它如何隨 n 縮小。
for n in [2, 10, 50, 250]:
    means = sampling_distribution_of_mean(
        sample_size=n, n_experiments=4000, seed=0)
    print(f'n={n:>3}: std of mean = {np.std(means):.4f}')

### 反思問題

1. 若有人給你一檔「過去一年平均日報酬為正」的策略，根據本週的內容，你會用哪些理由質疑「這代表它真的有正期望報酬」？

## 常見錯誤

- **把 LLN（收斂到哪）和 CLT（不確定性多大）混為一談。**
- **忘記設定隨機種子，導致模擬結果無法重現。**
- **用很小的樣本估計平均，卻當作精確的真值。**
- **看到樣本平均為正就認定期望值為正。**

## 完成本週後，你應該能做到什麼

- [ ] 能用模擬區分並解釋 LLN 與 CLT。
- [ ] 能計算並解讀樣本平均的標準誤。
- [ ] 能說明為什麼短期策略報酬的平均不可盡信。

## 參考與致謝

- 本 notebook 的所有解說、範例與習題皆為本專案**原創**撰寫。
- 推薦的外部學習資源請見 [`docs/resources.md`](../../docs/resources.md)。
- 數學與財務概念筆記請見 [`docs/math/`](../../docs/math/) 與 [`docs/finance/`](../../docs/finance/)。

### 隱私與免責聲明

- 本 notebook 不含任何真實個人資訊。
- 本 notebook 僅使用可重現的合成資料，不需要網路連線。
- 本 notebook 不對任何策略做出實際投資獲利的宣稱。